# Tutorial — Construtor de Modelos (`ModelSegmenterUI`)

`ModelSegmenter` é o segmentador **orientado a modelo** do `yggdrasil.credit_risk`. Diferente dos
segmentadores `lgd`/`pd` (que crescem uma árvore de bins direto nas features), aqui o fluxo é:
**analisar variáveis → selecionar → treinar um modelo (logística / ML) → segmentar o *score* do modelo
em ratings → validar e exportar**. Um mesmo objeto atende **classificação** (ex.: PD) e **regressão**
(ex.: LGD) via `task_type`.

Este tutorial é focado na **UI** (`ModelSegmenterUI`): um *workbench* de 5 abas em `ipywidgets`. Como um
notebook estático não registra cliques, cada aba é explicada e em seguida **reproduzida em código** pelo
segmentador interno `ui.seg` (e/ou pelos *handlers* `ui._on_*`, que são exatamente o que os botões chamam).

> **Contrato de dados:** um `pandas.DataFrame` com a coluna-alvo (`target`), features numéricas e/ou
> categóricas, e — opcionais — coluna de amostra (`amostra`: DES/OOT…) e coluna de data (`dt_ref`).
> Requer os extras `[ui]` (ipywidgets) para a interface; o núcleo roda sem ela.

## 0. Setup e dataset sintético

Geramos uma base de **classificação** (alvo binário) com 6 features numéricas, 1 categórica, coluna de
safra (`dt_ref`) e amostra `DES`/`OOT` — o mesmo formato usado nos testes do módulo.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from IPython.display import display   # autossuficiente fora do kernel (nbconvert etc.)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 40)
rng = np.random.default_rng(0)

n = 3000
X, y = make_classification(n_samples=n, n_features=6, n_informative=4,
                           weights=[0.75], random_state=0)
df = pd.DataFrame(X, columns=[f"feat_{i:02d}" for i in range(6)])
df["target"] = y                                            # alvo binário 0/1
df["feat_cat"] = rng.choice(["A", "B", "C", "D"], size=n, p=[.4, .3, .2, .1])  # categórica
df["dt_ref"] = rng.choice(pd.date_range("2023-01-01", periods=10, freq="MS"), size=n)
df["amostra"] = np.where(df["dt_ref"] >= pd.Timestamp("2023-08-01"), "OOT", "DES")
df.head()

## 1. Abrindo a UI (`ModelSegmenterUI`)

Instancie a UI com o mesmo contrato e exiba o objeto — o Jupyter chama `_ipython_display_()` e mostra o
*workbench* com 5 abas:

| Aba | Para quê |
|-----|----------|
| **① Variáveis** | ranking por IV, seleção (auto/manual) e categorização das candidatas |
| **② Análise de variáveis** | diagnóstico por variável: log-odds, distribuição, PSI, inversão |
| **③ Modelo** | escolher algoritmo (logística / RF / GBM), treinar, ver métricas e SHAP |
| **④ Ratings & Score** | segmentar o *score* em ratings e checar bad-rate / monotonia |
| **⑤ Validar & Exportar** | backtest, PSI, exportar, salvar/carregar e logar no MLflow |

`ui.seg` é o `ModelSegmenter` por trás da interface — usaremos ele para reproduzir cada aba em código.

In [ ]:
from yggdrasil.credit_risk.model import ModelSegmenterUI

ui = ModelSegmenterUI(
    df, target="target", task_type="classification",
    sample_col="amostra", ref_sample="DES", date_col="dt_ref",
)
ui   # exibe o workbench (ipywidgets). Alternativas: ui.display() ou display(ui.panel)

In [ ]:
seg = ui.seg   # o ModelSegmenter por trás da UI — todos os métodos abaixo saem daqui
type(seg).__name__, seg.task_type

## 2. Aba ① Variáveis — ranking e seleção

Na UI: o ranking por **IV** aparece na tabela; os sliders **IV mínimo** / **PSI máximo** + o botão
**Auto-selecionar** incluem/excluem candidatas; também dá para incluir/excluir manualmente e marcar a
*categoria* de cada variável (`manter`/`revisar`/`descartar`).

Em código, isso é `seg.variable_iv()` (ranking) e `seg.auto_select(...)` (seleção automática).

In [ ]:
seg.variable_iv()   # ranking: variavel, tipo, n_bins, iv, forca, tendencia, n_inversoes, psi_*, incluida, ...

In [ ]:
# Auto-seleção (botão "Auto-selecionar"). min_iv=0.0 não filtra por IV, mas a inclusão ainda
# respeita max_psi=0.25 (candidata com pior PSI > 0.25 sai); require_monotonic está desligado por padrão.
seg.auto_select(min_iv=0.0, max_psi=0.25)
seg.selected_features()   # features que entrarão no modelo

## 3. Aba ② Análise de variáveis — diagnóstico por variável

Na UI: escolhe-se a variável e a amostra e clica **Analisar variável**, gerando cartões de resumo + gráficos
(log-odds por bin, distribuição, série temporal, PSI por safra e inversão de risco entre amostras/safras).

Em código: `seg.variable_summary(feat)` e os `plot_variable_*`.

In [ ]:
feat = seg.variable_iv().iloc[0]["variavel"]   # variável de maior IV
print("Variável analisada:", feat)
seg.variable_summary(feat)                          # %missing, stats, IV, força, tendência, PSI por amostra

In [ ]:
seg.plot_variable_logodds(feat)   # barras (repr_%) + linha (log-odds/risco) por bin

In [ ]:
seg.plot_variable_inversion_by_sample(feat)   # risco por bin em cada amostra — cruzamentos = inversão

## 4. Aba ③ Modelo — treino e métricas

Na UI: escolhe-se o **algoritmo** no dropdown e clica **Treinar modelo**. O dropdown lista só os
algoritmos válidos para o `task_type` atual — nesta UI de **classificação**: *Regressão Logística*
(`"logistica"`, padrão), *Random Forest* (`"random_forest"`) e *Gradient Boosting* (`"gradient_boosting"`);
o *Linear* (`"linear"`) só aparece com `task_type="regression"`. O treino calcula o `score_` de todas as
linhas e mostra as **métricas** por amostra + gráficos (ROC, KS, distribuição do score). O botão
**Calcular SHAP** gera o beeswarm e a barra de importância.

O *handler* do botão de treino é `ui._on_fit(None)`; em código direto é `seg.fit(...)`.

In [ ]:
seg.fit("logistica")          # treina a regressão logística nas features selecionadas
seg.metrics()                  # por amostra: amostra, n, auc, gini, ks, accuracy, f1, ... (DES e OOT)

In [ ]:
seg.plot_roc()   # curva ROC (classificação)

In [ ]:
seg.plot_ks()    # curva KS (classificação)

In [ ]:
# SHAP. O botão "Calcular SHAP" gera o beeswarm + a barra (sample_size=800); shap_importance() é um
# extra disponível em código. Depende do pacote `shap` (já no core do yggdrasil, separado do extra [ui]).
display(seg.shap_importance().head(10))
seg.plot_shap_bar(max_display=10)

## 5. Aba ④ Ratings & Score — segmentando o *score*

Na UI: escolhe-se o **método** (`decis`, `quantil`, `arvore`, `optbin`), o nº de ratings e a **fusão
monotônica**, e clica **Gerar ratings**. O *score* do modelo é então segmentado em ratings, com tabela
(bad-rate por rating), distribuição e diagnóstico de inversão/monotonia.

Em código: `seg.build_ratings(...)` + `seg.rating_table()`.

In [ ]:
seg.build_ratings(method="quantil", n_ratings=10, monotonic_fusion=True)
seg.rating_table()   # por rating (em ordem): n, repr_%, event_rate por amostra

In [ ]:
seg.plot_rating_badrate()        # risco (event rate) por rating

In [ ]:
seg.plot_rating_distribution()   # distribuição (%) dos ratings por amostra

## 6. Aba ⑤ Validar & Exportar — backtest, PSI, export, persistência

Na UI: **Backtest** (previsto × realizado por safra), **PSI** da distribuição de ratings,
**Exportar DataFrame** (gera `ui.result` = base + colunas `score`/`rating`), **Salvar/Carregar**
(JSON + modelo `joblib`) e **Registrar no MLflow**.

In [ ]:
display(seg.psi())                # PSI da distribuição de ratings por amostra (não-DES)
seg.backtest("dt_ref")            # por safra: n, previsto_medio, realizado_medio, gap, status

In [ ]:
# Exportar: base + score + rating. Na UI isto preenche ui.result.
scored = seg.assign(col_score="score", col_rating="rating")
scored[["target", "amostra", "score", "rating"]].head()

In [ ]:
# Persistência: grava o JSON de config + <path>.model.joblib (o modelo) AO LADO, no mesmo destino.
# No Databricks use um caminho persistente (DBFS/Volumes), ex.: seg.save("/dbfs/tmp/modelo_seg.json") —
# o cwd do driver é efêmero. O load() precisa do MESMO destino (JSON + .model.joblib) e do mesmo df.
seg.save("modelo_seg.json")
# Recarregar em outra sessão (load() devolve um NOVO segmentador):
# from yggdrasil.credit_risk.model import ModelSegmenter
# seg2 = ModelSegmenter(df, target="target", task_type="classification", sample_col="amostra",
#                       ref_sample="DES", date_col="dt_ref", verbose=False).load("modelo_seg.json", df)

# Escorar dados novos (aplica modelo + ratings):
seg.predict(df.head(5))[["score", "rating"]]

In [ ]:
# MLflow (descomente em ambiente com tracking, ex.: Databricks):
# run_id = seg.log_to_mlflow(experiment="/Shared/model_segmenter", run_name="modelo_v1")
# print("run_id:", run_id)

## 7. Dirigindo a UI por código (cliques programáticos)

Cada botão da UI chama um *handler* `ui._on_*`. Dá para reproduzir o fluxo inteiro sem clicar — útil para
testes/automação. Eles operam sobre o **mesmo** `ui.seg`, então o estado fica refletido na interface.

In [ ]:
ui._on_fit(None)             # = clicar "Treinar"
ui._on_build_ratings(None)   # = clicar "Gerar ratings"
ui._on_export(None)          # = clicar "Exportar" -> preenche ui.result
ui.result.shape, ui.seg.score_ is not None

## 8. Notas

- **Classificação × regressão:** o mesmo `ModelSegmenter`/`ModelSegmenterUI` atende os dois via
  `task_type="classification"` (alvo 0/1; métricas AUC/Gini/KS) ou `"regression"` (alvo contínuo;
  métricas RMSE/MAE/R²; troque o algoritmo para `"linear"`/`"random_forest"`/`"gradient_boosting"`).
- **`ui.seg`** dá acesso direto a todos os métodos do segmentador; **`ui.result`** guarda a base exportada.
- **Sem interface:** o núcleo (`ModelSegmenter`) roda sem ipywidgets — use os métodos diretamente.
- Outros tutoriais (mesma pasta): `00_tutorial_yggdrasil`, `01_tutorial_lgd`, `02_tutorial_eda_features`,
  `03_tutorial_feature_selection`, `04_tutorial_lgd_segmenter`, `05_tutorial_pd_segmenter`,
  `07_tutorial_esteira_ml_mlflow`, `08_tutorial_validacao_lgd`.